In [2]:
"""
Generate a reproducible SQLite database for a fictional UK council's
urban tree inspection and maintenance records.

The data is synthetic and generated by this script; it is not downloaded.

Measurement types represented in the database:
- Nominal: tree species, site type, and defect type
- Ordinal: tree condition grade, ordered from 1 (critical) to 5 (excellent)
- Interval: soil temperature in Celsius
- Ratio: tree height, trunk diameter, canopy spread, work hours, and costs

Run with Python 3. No third-party packages are required.
"""

from datetime import date, timedelta
from pathlib import Path
import random
import sqlite3


# A fixed seed makes the same records every time the script is run.
SEED = 20261104
rng = random.Random(SEED)

# The fictional "Westbridge Council" keeps this database in the run folder.
DATABASE_PATH = Path.cwd() / "westbridge_urban_tree_health.sqlite"
AS_OF_DATE = date(2026, 11, 4)


DISTRICT_NAMES = [
    "North District",
    "South District",
    "East District",
    "West District",
    "Central District",
    "Riverside District",
    "Hilltop District",
    "Meadow District",
]

SPECIES = [
    ("English oak", "Quercus robur", 35.0, 0.35),
    ("Silver birch", "Betula pendula", 25.0, 0.55),
    ("Common lime", "Tilia x europaea", 30.0, 0.45),
    ("London plane", "Platanus x acerifolia", 35.0, 0.50),
    ("Norway maple", "Acer platanoides", 25.0, 0.40),
    ("Wild cherry", "Prunus avium", 20.0, 0.45),
    ("Rowan", "Sorbus aucuparia", 15.0, 0.35),
    ("Common ash", "Fraxinus excelsior", 30.0, 0.50),
    ("Field maple", "Acer campestre", 20.0, 0.30),
    ("Scots pine", "Pinus sylvestris", 30.0, 0.35),
    ("Sweet chestnut", "Castanea sativa", 30.0, 0.45),
    ("Crab apple", "Malus sylvestris", 12.0, 0.25),
    ("Hawthorn", "Crataegus monogyna", 10.0, 0.20),
    ("Hornbeam", "Carpinus betulus", 25.0, 0.35),
    ("Common beech", "Fagus sylvatica", 35.0, 0.40),
    ("White willow", "Salix alba", 25.0, 0.70),
]

CONDITION_GRADES = [
    (1, "Critical", "Immediate attention is required."),
    (2, "Poor", "Significant health or structural concerns are present."),
    (3, "Fair", "Some concerns are present; monitor or plan work."),
    (4, "Good", "The tree is generally healthy."),
    (5, "Excellent", "The tree is in very good condition."),
]

DEFECT_TYPES = [
    "Deadwood",
    "Branch damage",
    "Root damage",
    "Fungal growth",
    "Leaning trunk",
    "Pest damage",
]

MATERIALS = [
    ("Pruning supplies", 18.50),
    ("Tree stake", 12.00),
    ("Mulch", 8.75),
    ("Protective guard", 24.00),
    ("Soil improvement", 16.25),
    ("Cabling equipment", 42.00),
    ("Replacement sapling", 35.00),
    ("Pest treatment", 21.50),
    ("Root barrier", 48.00),
    ("Safety signage", 9.25),
]

SITE_TYPES = ["Park", "Roadside", "School ground", "Public garden"]
ROLES = ["Arborist", "Tree inspector", "Grounds officer"]


def random_date(start: date, end: date) -> date:
    """Choose a date between start and end, including both endpoints."""
    days_between = (end - start).days
    return start + timedelta(days=rng.randint(0, days_between))


def create_schema(connection: sqlite3.Connection) -> None:
    """Create the related tables and their integrity rules."""
    # Drop only tables owned by this generator, in child-to-parent order.
    tables_to_rebuild = [
        "inspection_imports",
        "work_order_materials",
        "work_order_crews",
        "work_orders",
        "inspections",
        "trees",
        "materials",
        "crews",
        "inspectors",
        "condition_grades",
        "species",
        "sites",
        "districts",
    ]
    for table_name in tables_to_rebuild:
        connection.execute(f"DROP TABLE IF EXISTS {table_name}")

    connection.executescript(
        """
        CREATE TABLE districts (
            district_id INTEGER PRIMARY KEY,
            district_name TEXT NOT NULL UNIQUE
        );

        CREATE TABLE sites (
            site_id INTEGER PRIMARY KEY,
            district_id INTEGER NOT NULL,
            site_code TEXT NOT NULL UNIQUE,
            site_name TEXT NOT NULL,
            site_type TEXT NOT NULL
                CHECK (site_type IN ('Park', 'Roadside', 'School ground', 'Public garden')),
            latitude REAL NOT NULL CHECK (latitude BETWEEN 50.0 AND 59.0),
            longitude REAL NOT NULL CHECK (longitude BETWEEN -7.0 AND 2.0),
            FOREIGN KEY (district_id) REFERENCES districts(district_id)
        );

        CREATE TABLE species (
            species_id INTEGER PRIMARY KEY,
            common_name TEXT NOT NULL UNIQUE,
            scientific_name TEXT NOT NULL UNIQUE,
            mature_height_m REAL NOT NULL CHECK (mature_height_m > 0),
            growth_rate_m_year REAL NOT NULL CHECK (growth_rate_m_year > 0)
        );

        CREATE TABLE condition_grades (
            grade INTEGER PRIMARY KEY CHECK (grade BETWEEN 1 AND 5),
            grade_label TEXT NOT NULL UNIQUE,
            description TEXT NOT NULL
        );

        CREATE TABLE trees (
            tree_id INTEGER PRIMARY KEY,
            site_id INTEGER NOT NULL,
            species_id INTEGER NOT NULL,
            tree_tag TEXT NOT NULL UNIQUE,
            planted_year INTEGER NOT NULL CHECK (planted_year BETWEEN 1900 AND 2025),
            is_protected INTEGER NOT NULL CHECK (is_protected IN (0, 1)),
            FOREIGN KEY (site_id) REFERENCES sites(site_id),
            FOREIGN KEY (species_id) REFERENCES species(species_id)
        );

        CREATE TABLE inspectors (
            inspector_id INTEGER PRIMARY KEY,
            staff_code TEXT NOT NULL UNIQUE,
            role TEXT NOT NULL
                CHECK (role IN ('Arborist', 'Tree inspector', 'Grounds officer'))
        );

        CREATE TABLE inspections (
            inspection_id INTEGER PRIMARY KEY,
            tree_id INTEGER NOT NULL,
            inspector_id INTEGER NOT NULL,
            inspected_on TEXT NOT NULL,
            condition_grade INTEGER NOT NULL CHECK (condition_grade BETWEEN 1 AND 5),
            defect_type TEXT
                CHECK (
                    defect_type IS NULL OR
                    defect_type IN (
                        'Deadwood',
                        'Branch damage',
                        'Root damage',
                        'Fungal growth',
                        'Leaning trunk',
                        'Pest damage'
                    )
                ),
            height_m REAL NOT NULL CHECK (height_m > 0),
            trunk_diameter_cm REAL NOT NULL CHECK (trunk_diameter_cm > 0),
            canopy_spread_m REAL NOT NULL CHECK (canopy_spread_m > 0),
            soil_temp_c REAL NOT NULL CHECK (soil_temp_c BETWEEN -10 AND 40),
            notes TEXT,
            FOREIGN KEY (tree_id) REFERENCES trees(tree_id),
            FOREIGN KEY (inspector_id) REFERENCES inspectors(inspector_id),
            FOREIGN KEY (condition_grade) REFERENCES condition_grades(grade)
        );

        CREATE TABLE crews (
            crew_id INTEGER PRIMARY KEY,
            crew_code TEXT NOT NULL UNIQUE,
            crew_specialism TEXT NOT NULL
                CHECK (crew_specialism IN ('Inspection', 'Pruning', 'Tree care', 'Removal'))
        );

        CREATE TABLE materials (
            material_id INTEGER PRIMARY KEY,
            material_name TEXT NOT NULL UNIQUE,
            standard_unit_cost_gbp REAL NOT NULL CHECK (standard_unit_cost_gbp >= 0)
        );

        CREATE TABLE work_orders (
            work_order_id INTEGER PRIMARY KEY,
            inspection_id INTEGER NOT NULL UNIQUE,
            priority_level INTEGER NOT NULL CHECK (priority_level BETWEEN 1 AND 4),
            status TEXT NOT NULL
                CHECK (status IN ('Open', 'In progress', 'Completed')),
            opened_on TEXT NOT NULL,
            completed_on TEXT,
            estimated_cost_gbp REAL NOT NULL CHECK (estimated_cost_gbp >= 0),
            summary TEXT NOT NULL,
            FOREIGN KEY (inspection_id) REFERENCES inspections(inspection_id),
            CHECK (completed_on IS NULL OR completed_on >= opened_on),
            CHECK (
                (status = 'Completed' AND completed_on IS NOT NULL) OR
                (status <> 'Completed' AND completed_on IS NULL)
            )
        );

        -- Each work order can have several crews, and each crew can handle
        -- several orders. The pair of IDs uniquely identifies each assignment.
        CREATE TABLE work_order_crews (
            work_order_id INTEGER NOT NULL,
            crew_id INTEGER NOT NULL,
            assigned_on TEXT NOT NULL,
            work_hours REAL NOT NULL CHECK (work_hours > 0),
            PRIMARY KEY (work_order_id, crew_id),
            FOREIGN KEY (work_order_id) REFERENCES work_orders(work_order_id),
            FOREIGN KEY (crew_id) REFERENCES crews(crew_id)
        );

        -- A work order can use several materials. The two IDs form a
        -- composite primary key so a material appears once per order.
        CREATE TABLE work_order_materials (
            work_order_id INTEGER NOT NULL,
            material_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL CHECK (quantity > 0),
            unit_cost_gbp REAL NOT NULL CHECK (unit_cost_gbp >= 0),
            PRIMARY KEY (work_order_id, material_id),
            FOREIGN KEY (work_order_id) REFERENCES work_orders(work_order_id),
            FOREIGN KEY (material_id) REFERENCES materials(material_id)
        );

        -- This table represents incoming inspection records before review.
        -- Repeated source_record_ref values are intentional duplicate imports.
        CREATE TABLE inspection_imports (
            import_id INTEGER PRIMARY KEY,
            inspection_id INTEGER NOT NULL,
            source_record_ref TEXT NOT NULL,
            received_on TEXT NOT NULL,
            source_defect_type TEXT,
            source_note TEXT,
            is_duplicate INTEGER NOT NULL CHECK (is_duplicate IN (0, 1)),
            FOREIGN KEY (inspection_id) REFERENCES inspections(inspection_id)
        );

        CREATE INDEX idx_inspections_tree_date
            ON inspections(tree_id, inspected_on);

        CREATE INDEX idx_work_orders_status
            ON work_orders(status);

        CREATE INDEX idx_imports_source_ref
            ON inspection_imports(source_record_ref);
        """
    )


def generate_database() -> None:
    """Generate the synthetic records and save them to the SQLite file."""
    connection = sqlite3.connect(DATABASE_PATH)
    connection.execute("PRAGMA foreign_keys = ON")

    try:
        create_schema(connection)

        # Reference tables keep repeated labels consistent across the database.
        connection.executemany(
            "INSERT INTO districts (district_name) VALUES (?)",
            [(name,) for name in DISTRICT_NAMES],
        )

        connection.executemany(
            """
            INSERT INTO species
                (common_name, scientific_name, mature_height_m, growth_rate_m_year)
            VALUES (?, ?, ?, ?)
            """,
            SPECIES,
        )

        connection.executemany(
            """
            INSERT INTO condition_grades (grade, grade_label, description)
            VALUES (?, ?, ?)
            """,
            CONDITION_GRADES,
        )

        connection.executemany(
            """
            INSERT INTO materials (material_name, standard_unit_cost_gbp)
            VALUES (?, ?)
            """,
            MATERIALS,
        )

        site_rows = []
        for site_number in range(1, 49):
            district_id = rng.randint(1, len(DISTRICT_NAMES))
            site_rows.append(
                (
                    district_id,
                    f"WB-SITE-{site_number:03d}",
                    f"{rng.choice(['Willow', 'Cedar', 'Meadow', 'Riverside', 'Kingfisher', 'Oak'])} "
                    f"{rng.choice(['Park', 'Green', 'Gardens', 'Common', 'Walk'])} {site_number:02d}",
                    rng.choice(SITE_TYPES),
                    round(rng.uniform(51.0, 57.5), 6),
                    round(rng.uniform(-5.5, 1.2), 6),
                )
            )

        connection.executemany(
            """
            INSERT INTO sites
                (district_id, site_code, site_name, site_type, latitude, longitude)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            site_rows,
        )

        inspector_rows = [
            (f"WB-INS-{number:03d}", rng.choice(ROLES))
            for number in range(1, 19)
        ]
        connection.executemany(
            "INSERT INTO inspectors (staff_code, role) VALUES (?, ?)",
            inspector_rows,
        )

        crew_specialisms = ["Inspection", "Pruning", "Tree care", "Removal"]
        connection.executemany(
            """
            INSERT INTO crews (crew_code, crew_specialism)
            VALUES (?, ?)
            """,
            [
                (f"WB-CREW-{number:02d}", rng.choice(crew_specialisms))
                for number in range(1, 11)
            ],
        )

        # Each tree is assigned to a real site and a known species.
        tree_rows = []
        tree_details = {}
        for tree_id in range(1, 651):
            site_id = rng.randint(1, len(site_rows))
            species_id = rng.randint(1, len(SPECIES))
            planted_year = rng.randint(1950, 2024)
            is_protected = 1 if rng.random() < 0.12 else 0
            tree_tag = f"WB-TREE-{tree_id:05d}"

            tree_rows.append(
                (site_id, species_id, tree_tag, planted_year, is_protected)
            )
            tree_details[tree_id] = {
                "species_id": species_id,
                "planted_year": planted_year,
            }

        connection.executemany(
            """
            INSERT INTO trees
                (site_id, species_id, tree_tag, planted_year, is_protected)
            VALUES (?, ?, ?, ?, ?)
            """,
            tree_rows,
        )

        # Species maturity and planting year influence tree measurements.
        inspection_rows = []
        inspection_by_id = {}
        missing_inspection_notes = 0
        start_date = date(2023, 1, 1)

        for inspection_id in range(1, 1801):
            tree_id = rng.randint(1, len(tree_rows))
            inspector_id = rng.randint(1, len(inspector_rows))
            inspected_on = random_date(start_date, AS_OF_DATE)
            tree = tree_details[tree_id]
            species_data = SPECIES[tree["species_id"] - 1]

            mature_height = species_data[2]
            growth_rate = species_data[3]
            age = max(1, inspected_on.year - tree["planted_year"])

            height_m = min(
                mature_height,
                max(2.0, age * growth_rate + rng.uniform(1.0, 5.0)),
            )
            trunk_diameter_cm = max(5.0, age * rng.uniform(0.35, 0.75))
            canopy_spread_m = max(1.5, height_m * rng.uniform(0.45, 0.85))

            # Older trees are somewhat more likely to receive a lower grade.
            grade_weights = [
                1.0 + age / 35,
                2.0 + age / 40,
                6.0,
                5.0,
                4.0,
            ]
            condition_grade = rng.choices(
                [1, 2, 3, 4, 5], weights=grade_weights, k=1
            )[0]

            # Defects are more common when the condition grade is low.
            if condition_grade <= 2:
                defect_type = rng.choice(DEFECT_TYPES) if rng.random() < 0.72 else None
            elif condition_grade == 3:
                defect_type = rng.choice(DEFECT_TYPES) if rng.random() < 0.35 else None
            else:
                defect_type = rng.choice(DEFECT_TYPES) if rng.random() < 0.10 else None

            # Celsius is an interval measure; the seasonal pattern avoids
            # assigning the same temperature distribution to every month.
            seasonal_temperature = 10 + 7 * rng.uniform(-1, 1)
            soil_temp_c = max(
                -2.0,
                min(28.0, seasonal_temperature + (inspected_on.month - 6) * 0.45),
            )

            if rng.random() < 0.12:
                notes = None
                missing_inspection_notes += 1
            else:
                notes = rng.choice(
                    [
                        "Routine visual inspection completed.",
                        "Canopy reviewed from ground level.",
                        "Site access was clear during inspection.",
                        "Follow-up observation recommended.",
                        "No urgent access issue was identified.",
                    ]
                )

            row = (
                tree_id,
                inspector_id,
                inspected_on.isoformat(),
                condition_grade,
                defect_type,
                round(height_m, 2),
                round(trunk_diameter_cm, 2),
                round(canopy_spread_m, 2),
                round(soil_temp_c, 1),
                notes,
            )
            inspection_rows.append(row)
            inspection_by_id[inspection_id] = {
                "tree_id": tree_id,
                "inspected_on": inspected_on,
                "condition_grade": condition_grade,
                "defect_type": defect_type,
                "notes": notes,
            }

        connection.executemany(
            """
            INSERT INTO inspections (
                tree_id, inspector_id, inspected_on, condition_grade,
                defect_type, height_m, trunk_diameter_cm, canopy_spread_m,
                soil_temp_c, notes
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            inspection_rows,
        )

        # Poor condition or a recorded defect can lead to a maintenance order.
        eligible_inspections = [
            inspection_id
            for inspection_id, record in inspection_by_id.items()
            if record["condition_grade"] <= 2
            or (
                record["condition_grade"] == 3
                and record["defect_type"] is not None
            )
        ]
        selected_inspections = rng.sample(
            eligible_inspections,
            k=min(420, len(eligible_inspections)),
        )

        work_order_rows = []
        crew_assignment_rows = []
        material_usage_rows = []
        next_work_order_id = 1

        for inspection_id in selected_inspections:
            inspection = inspection_by_id[inspection_id]
            condition_grade = inspection["condition_grade"]

            # Priority is ordinal: 1 is low and 4 is urgent.
            priority_level = {
                1: 4,
                2: 3,
                3: 2,
                4: 1,
                5: 1,
            }[condition_grade]

            inspected_on = inspection["inspected_on"]
            opened_on = min(
                inspected_on + timedelta(days=rng.randint(0, 14)),
                AS_OF_DATE,
            )
            days_available = max(0, (AS_OF_DATE - opened_on).days)

            if days_available > 0 and rng.random() < 0.78:
                status = "Completed"
                completed_on = (
                    opened_on
                    + timedelta(days=rng.randint(1, min(45, days_available)))
                )
                completed_text = completed_on.isoformat()
            else:
                status = rng.choice(["Open", "In progress"])
                completed_text = None

            number_of_materials = rng.randint(1, 3)
            chosen_materials = rng.sample(
                range(1, len(MATERIALS) + 1),
                k=number_of_materials,
            )

            material_total = 0.0
            for material_id in chosen_materials:
                material_name, standard_cost = MATERIALS[material_id - 1]
                quantity = rng.randint(1, 6)
                unit_cost = round(standard_cost * rng.uniform(0.9, 1.15), 2)
                material_total += quantity * unit_cost
                material_usage_rows.append(
                    (
                        next_work_order_id,
                        material_id,
                        quantity,
                        unit_cost,
                    )
                )

            number_of_crews = rng.randint(1, 2)
            chosen_crews = rng.sample(range(1, 11), k=number_of_crews)
            labour_total = 0.0
            for crew_id in chosen_crews:
                work_hours = round(rng.uniform(1.5, 7.5), 1)
                labour_total += work_hours * 28.0
                crew_assignment_rows.append(
                    (
                        next_work_order_id,
                        crew_id,
                        opened_on.isoformat(),
                        work_hours,
                    )
                )

            estimated_cost = round(65.0 + material_total + labour_total, 2)
            summary = (
                f"Inspect and address {inspection['defect_type'].lower()}."
                if inspection["defect_type"]
                else "Carry out planned tree health maintenance."
            )

            work_order_rows.append(
                (
                    inspection_id,
                    priority_level,
                    status,
                    opened_on.isoformat(),
                    completed_text,
                    estimated_cost,
                    summary,
                )
            )
            next_work_order_id += 1

        connection.executemany(
            """
            INSERT INTO work_orders (
                inspection_id, priority_level, status, opened_on,
                completed_on, estimated_cost_gbp, summary
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            work_order_rows,
        )

        connection.executemany(
            """
            INSERT INTO work_order_crews
                (work_order_id, crew_id, assigned_on, work_hours)
            VALUES (?, ?, ?, ?)
            """,
            crew_assignment_rows,
        )

        connection.executemany(
            """
            INSERT INTO work_order_materials
                (work_order_id, material_id, quantity, unit_cost_gbp)
            VALUES (?, ?, ?, ?)
            """,
            material_usage_rows,
        )

        # The original import is recorded once for each inspection. A small
        # number of rows are then copied to represent duplicate source records.
        # Missing source notes are retained as NULL and can be counted later.
        import_rows = []
        missing_source_notes = 0

        for inspection_id, record in inspection_by_id.items():
            if rng.random() < 0.12:
                source_note = None
                missing_source_notes += 1
            else:
                source_note = record["notes"]

            import_rows.append(
                (
                    inspection_id,
                    f"SOURCE-{inspection_id:05d}",
                    record["inspected_on"].isoformat(),
                    record["defect_type"],
                    source_note,
                    0,
                )
            )

        # These duplicate rows intentionally repeat the same source reference
        # and values. Their separate import IDs preserve the audit trail.
        duplicate_count = 36
        duplicate_source_rows = rng.sample(import_rows, k=duplicate_count)
        for original_row in duplicate_source_rows:
            duplicate_row = list(original_row)
            duplicate_row[5] = 1
            import_rows.append(tuple(duplicate_row))

        connection.executemany(
            """
            INSERT INTO inspection_imports (
                inspection_id, source_record_ref, received_on,
                source_defect_type, source_note, is_duplicate
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            import_rows,
        )

        connection.commit()

        # A quick summary makes the generated dataset easy to describe in the report.
        print(f"Database created: {DATABASE_PATH}")
        print(f"Random seed: {SEED}")
        print("\nTable row counts:")

        table_names = [
            "districts",
            "sites",
            "species",
            "condition_grades",
            "trees",
            "inspectors",
            "inspections",
            "crews",
            "materials",
            "work_orders",
            "work_order_crews",
            "work_order_materials",
            "inspection_imports",
        ]

        for table_name in table_names:
            row_count = connection.execute(
                f"SELECT COUNT(*) FROM {table_name}"
            ).fetchone()[0]
            print(f"  {table_name}: {row_count:,}")

        duplicate_rows = connection.execute(
            """
            SELECT COUNT(*)
            FROM inspection_imports
            WHERE is_duplicate = 1
            """
        ).fetchone()[0]

        print(f"\nInspections with deliberately missing notes: {missing_inspection_notes}")
        print(f"Imports with deliberately missing notes: {missing_source_notes}")
        print(f"Deliberate duplicate import rows: {duplicate_rows}")

        # This should return 1 when SQLite foreign-key enforcement is active.
        foreign_keys_enabled = connection.execute(
            "PRAGMA foreign_keys"
        ).fetchone()[0]
        print(f"Foreign-key enforcement enabled: {bool(foreign_keys_enabled)}")

    finally:
        connection.close()


if __name__ == "__main__":
    generate_database()

Database created: /content/westbridge_urban_tree_health.sqlite
Random seed: 20261104

Table row counts:
  districts: 8
  sites: 48
  species: 16
  condition_grades: 5
  trees: 650
  inspectors: 18
  inspections: 1,800
  crews: 10
  materials: 10
  work_orders: 420
  work_order_crews: 617
  work_order_materials: 834
  inspection_imports: 1,836

Inspections with deliberately missing notes: 202
Imports with deliberately missing notes: 202
Deliberate duplicate import rows: 36
Foreign-key enforcement enabled: True
